# Jaguar Re-ID Workflow on Google Colab
This notebook covers the full workflow from data preparation to inference. It mounts your Google Drive, updates the hardcoded paths in `config.py` to point to the Colab context, installs required dependencies, runs model training, and executes the re-ID procedure to output `submission.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Navigate to the directory where you uploaded the source code (the `.py` scripts) and data files (`train.csv`, `test.csv`):
Based on your Drive layout, it is at `/content/drive/MyDrive/re-idversion2`.

In [ ]:
import os

# Change this folder name here if it's different
project_dir = '/content/drive/MyDrive/re-idversion2'

if os.path.exists(project_dir):
    os.chdir(project_dir)
    print(f"Current working directory set to: {os.getcwd()}")
else:
    print(f"Failed to jump to {project_dir}, please check spelling or path.")

## 1. Setup Environment Paths (`config.py`)

The code uses a `config.py` file with hardcoded Windows paths. We'll automatically replace the directory paths in `config.py` to point to your current working Colab directory and image folders within `384x384`.

In [ ]:
current_dir = os.getcwd()
with open('config.py', 'r') as f:
    lines = f.readlines()

with open('config.py', 'w') as f:
    for line in lines:
        if "PROJECT_DIR =" in line:
            f.write(f"    PROJECT_DIR = r'{current_dir}'\n")
        elif "TRAIN_IMG_DIR =" in line:
            f.write(f"    TRAIN_IMG_DIR = os.path.join(PROJECT_DIR, '384x384', 'train-jaguars')\n")
        elif "TEST_IMG_DIR =" in line:
            f.write(f"    TEST_IMG_DIR = os.path.join(PROJECT_DIR, '384x384', 'test-jaguars')\n")
        else:
            f.write(line)
print(f"Successfully updated config.py PROJECT_DIR and image paths.")
!cat config.py | grep _DIR

## 2. Install Required Packages
Install `timm` for computer vision models, `imagehash` for perceptual hashing, and upgrade any standard data science libraries if necessary.

In [ ]:
!pip install -q timm imagehash

## 3. Train the Model
Runs `train.py`, carrying out cross-validation training on the `train-jaguars` image dataset.

In [ ]:
!python train.py

## 4. Run Inference & Generate Submission File
Run `main.py` which loads the model, extracts embeddings from `test-jaguars`, computes reranking similarity scores, and formats the output into `submission.csv`.

In [ ]:
!python main.py

## 5. Review Submission File
Finally, read `submission.csv` to ensure its layout looks correct to matches `sample_submission.csv` (`row_id` and `similarity` columns).

In [ ]:
import pandas as pd
from IPython.display import display

if os.path.exists('submission.csv'):
    sub_df = pd.read_csv('submission.csv')
    print(f"Generated Submission Size: {sub_df.shape}")
    display(sub_df.head())
else:
    print("\nCould not find 'submission.csv'. Verify that main.py ran without failures.")